# Feature-Conditioned Neural Network Correction for the 1D Euler Equations

This notebook tackles a **system** of conservation laws — the 1D Euler equations for
compressible gas dynamics.  Unlike the scalar Burgers case, the Euler equations have
three coupled variables ($\rho$, $\rho u$, $E$) and produce three distinct wave
structures: a rarefaction fan, a contact discontinuity, and a shock.

**Equations (conservative form):**
$$\frac{\partial}{\partial t} \begin{pmatrix} \rho \\ \rho u \\ E \end{pmatrix}
+ \frac{\partial}{\partial x} \begin{pmatrix} \rho u \\ \rho u^2 + p \\ u(E+p) \end{pmatrix} = 0$$

with equation of state $p = (\gamma-1)(E - \tfrac{1}{2}\rho u^2)$, $\gamma = 1.4$.

**Key idea:** The NN is conditioned on **physics-informed local features** — shock sensor,
Mach number, WENO smoothness indicators — so it can apply different corrections in
different regions of the flow (smooth rarefaction vs. sharp shock).

**Setup:**
- **Fine:** 800 points, Lax-Friedrichs + RK2
- **Coarse:** 100 points, same scheme (but under-resolved)
- **NN:** 13 features → 128 → 128 → 64 → 3 outputs (corrections to all conserved variables)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.dpi'] = 130

torch.manual_seed(42)
np.random.seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

## 1. Parameters and Grid

In [ ]:
GAMMA = 1.4

L   = 1.0
N_f = 800
N_c = 100
T   = 0.2
CFL = 0.4
ratio = N_f // N_c

x_f = np.linspace(0, L, N_f)
x_c = np.linspace(0, L, N_c)
dx_f = L / (N_f - 1)
dx_c = L / (N_c - 1)

print(f"Fine grid: {N_f} points, dx={dx_f:.5f}")
print(f"Coarse grid: {N_c} points, dx={dx_c:.5f}")
print(f"Refinement ratio: {ratio}x")

## 2. Euler Equation Utilities

We need conversions between **primitive** variables $(\rho, u, p)$ and **conserved**
variables $(\rho, \rho u, E)$, plus the flux function.

In [ ]:
def prim_to_cons(rho, u, p):
    E = p / (GAMMA - 1) + 0.5 * rho * u**2
    return np.stack([rho, rho * u, E], axis=0)

def cons_to_prim(U):
    rho = U[0]
    u   = U[1] / rho
    p   = (GAMMA - 1) * (U[2] - 0.5 * rho * u**2)
    p   = np.maximum(p, 1e-10)
    rho = np.maximum(rho, 1e-10)
    return rho, u, p

def euler_flux(U):
    rho, u, p = cons_to_prim(U)
    F = np.zeros_like(U)
    F[0] = rho * u
    F[1] = rho * u**2 + p
    F[2] = u * (U[2] + p)
    return F

def sound_speed(U):
    rho, u, p = cons_to_prim(U)
    return np.sqrt(GAMMA * p / rho)

## 3. Lax-Friedrichs Solver

The **global Lax-Friedrichs** scheme adds artificial viscosity proportional to the
maximum wave speed:

$$F_{j+1/2} = \frac{1}{2}\left[F(U_j) + F(U_{j+1})\right] - \frac{\alpha}{2}(U_{j+1} - U_j)$$

where $\alpha = \max_j(|u_j| + c_j)$ is the maximum characteristic speed.
This is simple and robust but very diffusive — exactly the kind of scheme that
benefits from a learned correction.

Time integration uses RK2 (Heun's method) with CFL-based adaptive time stepping.

In [ ]:
def lax_friedrichs_rhs(U, dx):
    F   = euler_flux(U)
    lam = np.max(np.abs(U[1]/U[0]) + sound_speed(U))
    F_half = 0.5 * (F[:, 1:] + F[:, :-1]) - 0.5 * lam * (U[:, 1:] - U[:, :-1])
    rhs = np.zeros_like(U)
    rhs[:, 1:-1] = -(F_half[:, 1:] - F_half[:, :-1]) / dx
    return rhs

def simulate(U0, dx, T, CFL=0.4):
    U = U0.copy()
    t = 0.0
    while t < T - 1e-12:
        c   = sound_speed(U)
        lam = np.max(np.abs(U[1]/U[0]) + c)
        dt  = min(CFL * dx / lam, T - t)
        # RK2 (Heun)
        k1  = lax_friedrichs_rhs(U, dx)
        U1  = U + dt * k1
        k2  = lax_friedrichs_rhs(U1, dx)
        U   = 0.5 * (U + U1 + dt * k2)
        t  += dt
    return U

def coarsen_U(U_fine, ratio):
    """Average conserved variables onto coarse grid."""
    N_f = U_fine.shape[1]
    N_c = N_f // ratio
    return U_fine[:, :N_c*ratio].reshape(3, N_c, ratio).mean(axis=2)

## 4. The Sod Shock Tube

The classic test case: a membrane at $x = 0.5$ separates high-pressure gas
(left: $\rho=1$, $p=1$) from low-pressure gas (right: $\rho=0.125$, $p=0.1$).
When the membrane breaks, three waves emerge:

1. **Rarefaction fan** (left-moving) — smooth expansion wave
2. **Contact discontinuity** (right-moving) — density jump, no pressure jump
3. **Shock wave** (right-moving) — sharp compression

Each of these requires a different type of correction, which is why we need
physics-informed features.

In [ ]:
def sod_ic(x):
    N  = len(x)
    rho = np.where(x < 0.5, 1.0, 0.125)
    u   = np.zeros(N)
    p   = np.where(x < 0.5, 1.0, 0.1)
    return prim_to_cons(rho, u, p)

# Visualise the initial condition and the fine solution
U0_demo_f = sod_ic(x_f)
U_demo_f  = simulate(U0_demo_f, dx_f, T, CFL)
U0_demo_c = coarsen_U(U0_demo_f, ratio)
U_demo_c  = simulate(U0_demo_c, dx_c, T, CFL)
U_demo_fc = coarsen_U(U_demo_f, ratio)

rho_f, u_f, p_f = cons_to_prim(U_demo_f)
rho_c, u_c, p_c = cons_to_prim(U_demo_c)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, var_f, var_c, name in zip(axes,
    [rho_f, u_f, p_f], [rho_c, u_c, p_c],
    ['Density $\\rho$', 'Velocity $u$', 'Pressure $p$']):
    ax.plot(x_f, var_f, 'k-', lw=2, label='Fine (800 pts)')
    ax.plot(x_c, var_c, 'r--', lw=1.8, label='Coarse (100 pts)')
    ax.set_title(name); ax.set_xlabel('x')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.suptitle(f'Sod Shock Tube at T={T}', fontsize=13)
plt.tight_layout()
plt.show()

print("The coarse solver smears all three wave structures.")
print("The shock and contact discontinuity are most affected.")

## 5. Physics-Informed Features

The key innovation for systems is **feature engineering** — instead of feeding raw
conserved variables, we compute physics-informed quantities that help the network
distinguish different flow regions:

| Feature | Formula | What it detects |
|---------|---------|----------------|
| $\rho, u, p$ | Primitive variables | Basic flow state |
| $M = |u|/c$ | Mach number | Compressibility regime |
| $\rho_{j\pm1}, u_{j\pm1}, p_{j\pm1}$ | Neighbours | Local gradients |
| $\Phi^s$ | Dilatation shock sensor | Compression regions (shocks) |
| $\beta_\rho, \beta_p$ | WENO smoothness indicators | Solution regularity |

Total: **13 features**.  The shock sensor $\Phi^s$ is large at shocks but zero in
smooth regions; the WENO $\beta$ indicators measure solution smoothness using
the standard WENO-5 polynomial oscillation measure.

In [ ]:
def _pad(u):
    """Constant extrapolation at boundaries (ghost cells)."""
    return np.concatenate([[u[0], u[0]], u, [u[-1], u[-1]]])

def weno_smoothness(u, dx):
    """Scalar WENO smoothness indicator (average of 3 sub-stencil betas)."""
    up = _pad(u)  # length N+4
    N = len(u)
    um2 = up[:N]; um1 = up[1:N+1]; uc = up[2:N+2]; up1 = up[3:N+3]; up2 = up[4:N+4]
    b0 = (13/12*(um2 - 2*um1 + uc)**2 + 0.25*(um2 - 4*um1 + 3*uc)**2)
    b1 = (13/12*(um1 - 2*uc + up1)**2 + 0.25*(um1 - up1)**2)
    b2 = (13/12*(uc - 2*up1 + up2)**2 + 0.25*(3*uc - 4*up1 + up2)**2)
    return (b0 + b1 + b2) / 3.0

def shock_sensor(u, dx):
    """Dilatation-based shock sensor (positive at compressions)."""
    up = _pad(u)
    N = len(u)
    return np.maximum(0, -(up[3:N+3] - up[1:N+1]) / (2*dx))

def extract_features_euler(U, dx):
    """
    Extract features for each grid point using ghost-cell padding
    (constant extrapolation) instead of periodic wrapping.
    Returns array of shape (N, 13).
    """
    rho, u, p = cons_to_prim(U)
    c   = np.sqrt(GAMMA * p / rho)
    M   = np.abs(u) / c

    beta_rho = weno_smoothness(rho, dx)
    beta_p   = weno_smoothness(p, dx)
    phi_s    = shock_sensor(u, dx)

    # Neighbour values via ghost-cell padding
    rho_p = _pad(rho); u_p = _pad(u); p_p = _pad(p)
    N = len(rho)

    features = np.stack([
        rho, u, p, M,
        rho_p[3:N+3], rho_p[1:N+1],   # rho_{j+1}, rho_{j-1}
        u_p[3:N+3],   u_p[1:N+1],     # u_{j+1},   u_{j-1}
        p_p[3:N+3],   p_p[1:N+1],     # p_{j+1},   p_{j-1}
        phi_s, beta_rho, beta_p
    ], axis=1)
    return features.astype(np.float32)

# Number of boundary points to exclude from training
N_GHOST = 3

Let's visualise the features for the Sod problem to build intuition:

In [ ]:
feat_demo = extract_features_euler(U_demo_c, dx_c)
rho_d, u_d, p_d = cons_to_prim(U_demo_c)

fig, axes = plt.subplots(2, 2, figsize=(12, 7))

ax = axes[0, 0]
ax.plot(x_c, rho_d, 'k-', label='$\\rho$')
ax.plot(x_c, p_d, 'b--', label='$p$')
ax.set_title('Primitive variables'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ax.plot(x_c, feat_demo[:, 3], 'g-', lw=1.5)
ax.set_title('Mach number $M = |u|/c$'); ax.grid(True, alpha=0.3)

ax = axes[1, 0]
ax.plot(x_c, feat_demo[:, 10], 'r-', lw=1.5)
ax.set_title('Shock sensor $\\Phi^s$'); ax.grid(True, alpha=0.3)
ax.set_xlabel('x')

ax = axes[1, 1]
ax.semilogy(x_c, feat_demo[:, 11] + 1e-15, 'b-', label='$\\beta_\\rho$')
ax.semilogy(x_c, feat_demo[:, 12] + 1e-15, 'r--', label='$\\beta_p$')
ax.set_title('WENO smoothness $\\beta$'); ax.legend(); ax.grid(True, alpha=0.3)
ax.set_xlabel('x')

plt.suptitle('Physics-informed features for the Sod problem', fontsize=12)
plt.tight_layout()
plt.show()

print("The shock sensor peaks at the shock; WENO beta spikes at discontinuities.")
print("These give the network information about WHERE corrections are needed.")

## 6. Training Data: Random Riemann Problems

We generate 230 random Riemann problems (200 train, 30 test) by randomising the
left/right states and the membrane location.  Each produces a different combination
of rarefaction, contact, and shock waves — the network must learn to handle all three.

In [ ]:
def random_riemann_ic(x, rng):
    rho_L = rng.uniform(0.5, 2.0)
    rho_R = rng.uniform(0.1, 0.5)
    u_L   = rng.uniform(-0.5, 0.5)
    u_R   = rng.uniform(-0.5, 0.5)
    p_L   = rng.uniform(0.5, 2.0)
    p_R   = rng.uniform(0.05, 0.3)
    x_d   = rng.uniform(0.3, 0.7)
    rho = np.where(x < x_d, rho_L, rho_R)
    u   = np.where(x < x_d, u_L,   u_R)
    p   = np.where(x < x_d, p_L,   p_R)
    return prim_to_cons(rho, u, p)

N_IC   = 200
N_test = 30
rng    = np.random.default_rng(7)

In [ ]:
print("Generating dataset (~3 min) ...")
X_list, Y_list = [], []

for i in range(N_IC + N_test):
    U0_f = random_riemann_ic(x_f, rng)
    U_f  = simulate(U0_f, dx_f, T, CFL)
    U0_c = coarsen_U(U0_f, ratio)
    U_c  = simulate(U0_c, dx_c, T, CFL)
    U_f_coarsened = coarsen_U(U_f, ratio)

    feat   = extract_features_euler(U_c, dx_c)
    target = (U_f_coarsened - U_c).T

    # Exclude boundary points where ghost-cell padding is unreliable
    feat   = feat[N_GHOST:-N_GHOST]
    target = target[N_GHOST:-N_GHOST]

    X_list.append(feat)
    Y_list.append(target)
    if (i+1) % 50 == 0:
        print(f"  {i+1}/{N_IC+N_test}")

X_all = np.concatenate([x[np.newaxis] for x in X_list[:N_IC]], axis=0).astype(np.float32)
Y_all_arr = np.concatenate([y[np.newaxis] for y in Y_list[:N_IC]], axis=0).astype(np.float32)
X_test_all = np.concatenate([x[np.newaxis] for x in X_list[N_IC:]], axis=0).astype(np.float32)
Y_test_all = np.concatenate([y[np.newaxis] for y in Y_list[N_IC:]], axis=0).astype(np.float32)

N_interior = N_c - 2 * N_GHOST

# Flatten: each interior grid point is a sample
X_tr_raw = X_all.reshape(-1, 13)
Y_tr_raw = Y_all_arr.reshape(-1, 3)
X_te_raw = X_test_all.reshape(-1, 13)
Y_te_raw = Y_test_all.reshape(-1, 3)

# Feature normalization (zero-mean, unit-variance)
feat_mean = X_tr_raw.mean(axis=0)
feat_std  = X_tr_raw.std(axis=0) + 1e-8   # avoid division by zero

X_tr_norm = (X_tr_raw - feat_mean) / feat_std
X_te_norm = (X_te_raw - feat_mean) / feat_std

X_tr = torch.tensor(X_tr_norm).to(DEVICE)
Y_tr = torch.tensor(Y_tr_raw).to(DEVICE)
X_te = torch.tensor(X_te_norm).to(DEVICE)
Y_te = torch.tensor(Y_te_raw).to(DEVICE)

print(f"\nInterior points per IC: {N_interior}  (excluded {2*N_GHOST} boundary pts)")
print(f"Training samples: {X_tr.shape[0]}  (= {N_IC} ICs × {N_interior} points)")
print(f"Test samples:     {X_te.shape[0]}")
print(f"\nFeature stats (training set):")
for i, name in enumerate(['rho','u','p','M','rho+','rho-','u+','u-','p+','p-','phi_s','beta_rho','beta_p']):
    print(f"  {name:8s}: mean={feat_mean[i]:+.3e}  std={feat_std[i]:.3e}")

## 7. The Correction Network

This network outputs **3 values** (corrections to $\rho$, $\rho u$, $E$) from
13 input features.  The architecture is 13 → 128 → 128 → 64 → 3.

In [ ]:
class EulerCorrectionNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(13, 128), nn.Tanh(),
            nn.Linear(128, 128), nn.Tanh(),
            nn.Linear(128, 64),  nn.Tanh(),
            nn.Linear(64, 3)
        )

    def forward(self, x):
        return self.net(x)

EPOCHS = 800
BATCH_SIZE = 512

model     = EulerCorrectionNet().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, EPOCHS)
loss_fn   = nn.MSELoss()

n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}")
print(f"Epochs: {EPOCHS}, Batch size: {BATCH_SIZE}")
print(model)

## 8. Training

In [ ]:
print("Training ...")
train_losses, test_losses = [], []

dataset = torch.utils.data.TensorDataset(X_tr, Y_tr)
loader  = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0
    for xb, yb in loader:
        optimizer.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(xb)
    scheduler.step()
    epoch_loss /= len(X_tr)

    model.eval()
    with torch.no_grad():
        test_loss = loss_fn(model(X_te), Y_te).item()
    train_losses.append(epoch_loss)
    test_losses.append(test_loss)

    if (epoch+1) % 100 == 0:
        print(f"  Epoch {epoch+1}  train={epoch_loss:.2e}  test={test_loss:.2e}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.semilogy(train_losses, label='Train', alpha=0.7)
ax.semilogy(test_losses,  label='Test',  alpha=0.7)
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
ax.set_title('Training convergence')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Evaluation: Sod Shock Tube

We test on the standard Sod problem — the membrane at $x=0.5$ with the classical
left/right states.  Note that the Sod IC was **not** in the training set (which used
randomised Riemann problems), so this tests generalisation.

In [ ]:
model.eval()

U0_f = sod_ic(x_f)
U0_c = coarsen_U(U0_f, ratio)

U_f  = simulate(U0_f, dx_f, T, CFL)
U_c  = simulate(U0_c, dx_c, T, CFL)
U_fc = coarsen_U(U_f, ratio)

feat = extract_features_euler(U_c, dx_c)
# Apply the same normalization used during training
feat_norm = (feat - feat_mean) / feat_std
with torch.no_grad():
    delta = model(torch.tensor(feat_norm).to(DEVICE)).cpu().numpy()

U_corr = U_c.copy()
# Only apply correction to interior points
U_corr[:, N_GHOST:-N_GHOST] = U_c[:, N_GHOST:-N_GHOST] + delta[N_GHOST:-N_GHOST].T

## 10. Results

**Top row:** solutions for each conserved variable (density, momentum, energy).
**Bottom row:** pointwise error before and after correction.

In [ ]:
var_names = ['Density $\\rho$', 'Momentum $\\rho u$', 'Energy $E$']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

sl = slice(N_GHOST, -N_GHOST)  # interior slice for RMSE

for v in range(3):
    ax = axes[0, v]
    ax.plot(x_c, U_fc[v],   'k-',  lw=2,   label='Fine (truth)')
    ax.plot(x_c, U_c[v],    'r--', lw=1.8, label='Coarse LxF')
    ax.plot(x_c, U_corr[v], 'b-',  lw=1.8, label='Coarse + NN')
    ax.set_title(var_names[v])
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    ax = axes[1, v]
    ax.plot(x_c, U_fc[v] - U_c[v],    'r--', label='Error before')
    ax.plot(x_c, U_fc[v] - U_corr[v], 'b-',  label='Error after')
    ax.axhline(0, color='k', lw=0.8)
    ax.set_xlabel('x'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

fig.suptitle('Sod Shock Tube: Feature-Based NN Correction', fontsize=14)
plt.tight_layout()
plt.show()

for v, name in enumerate(['rho', 'rho*u', 'E']):
    rb = np.mean((U_fc[v, sl]-U_c[v, sl])**2)**0.5
    ra = np.mean((U_fc[v, sl]-U_corr[v, sl])**2)**0.5
    print(f"  {name}: RMSE {rb:.3e} -> {ra:.3e}  ({rb/ra:.1f}x improvement)")

## 11. Stress Test: Stronger Shock

Let's test with a stronger pressure ratio (10:0.05 instead of 1:0.1) and
non-zero initial velocities — a harder Riemann problem not seen in training.

In [ ]:
# Strong shock test
rho_strong = np.where(x_f < 0.4, 2.0, 0.1)
u_strong   = np.where(x_f < 0.4, 0.3, -0.2)
p_strong   = np.where(x_f < 0.4, 10.0, 0.05)
U0_strong_f = prim_to_cons(rho_strong, u_strong, p_strong)

U_strong_f  = simulate(U0_strong_f, dx_f, T, CFL)
U0_strong_c = coarsen_U(U0_strong_f, ratio)
U_strong_c  = simulate(U0_strong_c, dx_c, T, CFL)
U_strong_fc = coarsen_U(U_strong_f, ratio)

feat_strong = extract_features_euler(U_strong_c, dx_c)
feat_strong_norm = (feat_strong - feat_mean) / feat_std
with torch.no_grad():
    delta_strong = model(torch.tensor(feat_strong_norm).to(DEVICE)).cpu().numpy()
U_strong_corr = U_strong_c.copy()
U_strong_corr[:, N_GHOST:-N_GHOST] = U_strong_c[:, N_GHOST:-N_GHOST] + delta_strong[N_GHOST:-N_GHOST].T

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for v, name in enumerate(['$\\rho$', '$\\rho u$', '$E$']):
    ax = axes[v]
    ax.plot(x_c, U_strong_fc[v],   'k-',  lw=2,   label='Fine')
    ax.plot(x_c, U_strong_c[v],    'r--', lw=1.8, label='Coarse')
    ax.plot(x_c, U_strong_corr[v], 'b-',  lw=1.8, label='Coarse + NN')
    ax.set_title(f'Strong shock: {name}'); ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3); ax.set_xlabel('x')

plt.suptitle('Stress test: strong Riemann problem (out of training range)', fontsize=12)
plt.tight_layout()
plt.show()

for v, name in enumerate(['rho', 'rho*u', 'E']):
    rb = np.mean((U_strong_fc[v, sl]-U_strong_c[v, sl])**2)**0.5
    ra = np.mean((U_strong_fc[v, sl]-U_strong_corr[v, sl])**2)**0.5
    ratio_str = f"{rb/ra:.1f}x better" if ra < rb else f"{ra/rb:.1f}x worse"
    print(f"  {name}: RMSE {rb:.3e} -> {ra:.3e}  ({ratio_str})")

## 12. Feature Importance: What Does the Network Use?

We compute a simple sensitivity measure: for each feature, we measure how much
the network's output changes when that feature is perturbed by $\pm\sigma$
(one standard deviation of the training data).  Features that the network
relies on heavily will show large output variation.

In [ ]:
feature_labels = ['$\\rho$', '$u$', '$p$', '$M$',
                  '$\\rho_{+1}$', '$\\rho_{-1}$',
                  '$u_{+1}$', '$u_{-1}$',
                  '$p_{+1}$', '$p_{-1}$',
                  '$\\Phi^s$', '$\\beta_\\rho$', '$\\beta_p$']

# Compute per-feature sensitivity on normalized inputs
# Perturb by 1 sigma in normalized space (= 1 unit)
X_mean_norm = X_tr.mean(dim=0)

sensitivities = []
with torch.no_grad():
    base_out = model(X_mean_norm.unsqueeze(0))
    for i in range(13):
        perturbed = X_mean_norm.clone()
        perturbed[i] += 1.0   # +1 sigma in normalised space
        pert_out = model(perturbed.unsqueeze(0))
        sensitivities.append(torch.abs(pert_out - base_out).sum().item())

sensitivities = np.array(sensitivities)
sensitivities /= sensitivities.max()  # normalise

fig, ax = plt.subplots(figsize=(10, 4))
colors = ['#d62728' if s > 0.5 else '#1f77b4' for s in sensitivities]
ax.barh(range(13), sensitivities, color=colors, alpha=0.8)
ax.set_yticks(range(13))
ax.set_yticklabels(feature_labels)
ax.set_xlabel('Normalised sensitivity')
ax.set_title('Feature importance (perturbation sensitivity)')
ax.grid(True, alpha=0.3, axis='x')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Summary

The Euler equations demonstrate several key points about learned corrections for systems:

1. **Multi-output corrections.** The network outputs corrections to all three conserved
   variables simultaneously — the coupling between $\rho$, $\rho u$, and $E$ is
   learned implicitly.

2. **Physics-informed features matter.** The shock sensor and WENO smoothness indicators
   give the network structural information about where corrections are needed, leading to
   spatially appropriate corrections (large at shocks, small in smooth regions).

3. **Training on diverse Riemann problems generalises to the standard Sod case.** The
   network never saw the exact Sod initial condition during training, yet it produces
   useful corrections.

4. **Out-of-distribution limits exist.** Very strong shocks (outside the training range)
   may not be corrected well — this is a fundamental limitation of a priori training
   that a posteriori methods (notebook 04) can partially address.